# Sección 2.A: Pre-Procesamiento de datos

#### Transformar las variables categóricas codificadas numéricamente utilizando "Codificación One-Hot EnCoding", luego Re-ordenar al final del dataset y Re-nombrar la variable dependiente o feature objetivo: "***default payment next month***"
    
Reemplazaremos los espacios en el nombre del feature, por guiones bajos para prevenir futuros errores a la hora de entrenar el modelo.
También moveremos al final del dataset el feature objetivo en este caso luego de haber modificado su nombre sería: "**default_payment_next_month**"

In [1]:
import pandas as pd
import os

# 1. Detectar la raíz del proyecto
# Si el notebook está en 'B_notebooks', subimos un nivel para llegar a la carpeta principal
directorio_actual = os.getcwd()
raiz_proyecto = os.path.dirname(directorio_actual)

# 2. Construir rutas dinámicas y robustas
# Esto une las carpetas de forma segura según el Sistema Operativo
ruta_lectura = os.path.join(raiz_proyecto, 'A_data', '01_raw', 'default_of_credit_card_clients.csv')
ruta_guardado = os.path.join(raiz_proyecto, 'A_data', '02_processed', 'processed_credit_card_data.csv')

print(f"📂 Buscando archivo en: {ruta_lectura}")

try:
    # 3. Carga del dataset
    # Usamos header=1 según tu análisis de que los nombres reales están en la 2da fila
    df = pd.read_csv(ruta_lectura, header=1)
    print("✅ Dataset cargado con éxito.")

    # --- FASE: FEATURE ENGINEERING ---
    # Creamos las variables antes de transformar las categóricas
    df['utilization_ratio'] = df['BILL_AMT1'] / df['LIMIT_BAL']
    
    bill_cols = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
    df['avg_bill_amt'] = df[bill_cols].mean(axis=1)
    
    pay_status_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
    exist_pay_cols = [c for c in pay_status_cols if c in df.columns]
    df['max_payment_delay'] = df[exist_pay_cols].max(axis=1)
    
    df['debt_growth'] = df['BILL_AMT1'] - df['avg_bill_amt']

    # --- FASE: ONE-HOT ENCODING ---
    mappings = {
        'SEX': {1: 'MAN', 2: 'WOMAN'},
        'EDUCATION': {1: 'GRAD_SCHOOL', 2: 'UNIVERSITY', 3: 'HIGH_SCHOOL', 4: 'OTHERS'},
        'MARRIAGE': {1: 'MARRIED', 2: 'SINGLE', 3: 'OTHERS'}
    }

    for col, mapping in mappings.items():
        if col in df.columns:
            dummies = pd.get_dummies(df[col].map(mapping), prefix=col, dtype=int)
            df = pd.concat([df, dummies], axis=1)

    # Limpieza de columnas originales e ID
    columnas_a_eliminar = ['ID', 'SEX', 'EDUCATION', 'MARRIAGE']
    df.drop(columns=[c for c in columnas_a_eliminar if c in df.columns], inplace=True)

    # --- FASE: ORDENAMIENTO ESTÉTICO ---
    target_viejo = 'default payment next month'
    target_nuevo = 'default_payment_next_month'
    
    if target_viejo in df.columns:
        df.rename(columns={target_viejo: target_nuevo}, inplace=True)
    
    # Reordenar: Todo menos el Target + Target al final
    cols_sin_target = [c for c in df.columns if c != target_nuevo]
    df = df[cols_sin_target + [target_nuevo]]

    # 4. Guardar resultado final
    # Creamos la carpeta de destino si no existe por seguridad
    os.makedirs(os.path.dirname(ruta_guardado), exist_ok=True)
    df.to_csv(ruta_guardado, index=False)
    
    print(f"🚀 Proceso completado. Archivo guardado en: {ruta_guardado}")

except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo. Verifica que la carpeta 'data/01_raw' esté en la raíz.")
except Exception as e:
    print(f"⚠️ Error inesperado: {e}")

# Vista previa
df.head()

📂 Buscando archivo en: c:\Users\mbeni\Desktop\Portafolio_Projects\default_of_credit_card_clients\A_data\01_raw\default_of_credit_card_clients.csv
✅ Dataset cargado con éxito.


🚀 Proceso completado. Archivo guardado en: c:\Users\mbeni\Desktop\Portafolio_Projects\default_of_credit_card_clients\A_data\02_processed\processed_credit_card_data.csv


,LIMIT_BAL,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,...,SEX_MAN,SEX_WOMAN,EDUCATION_GRAD_SCHOOL,EDUCATION_HIGH_SCHOOL,EDUCATION_OTHERS,EDUCATION_UNIVERSITY,MARRIAGE_MARRIED,MARRIAGE_OTHERS,MARRIAGE_SINGLE,default_payment_next_month
0,20000,24,2,2,-1,-1,-2,-2,3913,3102,...,0,1,0,0,0,1,1,0,0,1
1,120000,26,-1,2,0,0,0,2,2682,1725,...,0,1,0,0,0,1,0,0,1,1
2,90000,34,0,0,0,0,0,0,29239,14027,...,0,1,0,0,0,1,0,0,1,0
3,50000,37,0,0,0,0,0,0,46990,48233,...,0,1,0,0,0,1,1,0,0,0
4,50000,57,-1,0,-1,0,0,0,8617,5670,...,1,0,0,0,0,1,1,0,0,0


# Sección 2.B: División de Datos y Balanceo

In [2]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import numpy as np

# Separar features y target
X = df.drop('default_payment_next_month', axis=1)
y = df['default_payment_next_month']

print(f"📊 Dataset completo:")
print(f"   Tamaño: {X.shape}")
print(f"   Distribución de clases:")
print(f"   - No default (0): {(y==0).sum()} ({(y==0).sum()/len(y)*100:.2f}%)")
print(f"   - Default (1): {(y==1).sum()} ({(y==1).sum()/len(y)*100:.2f}%)")

# Split train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✂️ División Train/Test:")
print(f"   Train: {X_train.shape} | Test: {X_test.shape}")

# Aplicar SMOTE solo al conjunto de entrenamiento
print(f"\n⚖️ Aplicando SMOTE al conjunto de entrenamiento...")
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"✅ Balanceo completado:")
print(f"   Train original: {X_train.shape}")
print(f"   Train balanceado: {X_train_balanced.shape}")
print(f"   Distribución balanceada:")
print(f"   - No default (0): {(y_train_balanced==0).sum()}")
print(f"   - Default (1): {(y_train_balanced==1).sum()}")

# Guardar todos los conjuntos
output_dir = os.path.join(raiz_proyecto, 'A_data', '02_processed')
os.makedirs(output_dir, exist_ok=True)

np.save(os.path.join(output_dir, 'X_train.npy'), X_train.values)
np.save(os.path.join(output_dir, 'X_test.npy'), X_test.values)
np.save(os.path.join(output_dir, 'y_train.npy'), y_train.values)
np.save(os.path.join(output_dir, 'y_test.npy'), y_test.values)
np.save(os.path.join(output_dir, 'X_train_balanced.npy'), X_train_balanced)
np.save(os.path.join(output_dir, 'y_train_balanced.npy'), y_train_balanced)

print(f"\n💾 Archivos guardados en: {output_dir}")
print("   - X_train.npy, X_test.npy")
print("   - y_train.npy, y_test.npy")
print("   - X_train_balanced.npy, y_train_balanced.npy")
print("\n✅ Preprocesamiento completado exitosamente!")

📊 Dataset completo:
   Tamaño: (30000, 33)
   Distribución de clases:
   - No default (0): 23364 (77.88%)
   - Default (1): 6636 (22.12%)

✂️ División Train/Test:
   Train: (24000, 33) | Test: (6000, 33)

⚖️ Aplicando SMOTE al conjunto de entrenamiento...
✅ Balanceo completado:
   Train original: (24000, 33)
   Train balanceado: (37382, 33)
   Distribución balanceada:
   - No default (0): 18691
   - Default (1): 18691



💾 Archivos guardados en: c:\Users\mbeni\Desktop\Portafolio_Projects\default_of_credit_card_clients\A_data\02_processed
   - X_train.npy, X_test.npy
   - y_train.npy, y_test.npy
   - X_train_balanced.npy, y_train_balanced.npy

✅ Preprocesamiento completado exitosamente!
